In [ ]:
import datetime
from datetime import timedelta
import os
import time
import pandas as pd
import numpy as np
import yfinance as yf
import random
from glob import glob
from fidelity_option_data_downloader import FidelityOptionDataDownloader, get_rotating_logger
from option_analyzer import OptionAnalyzer
from indicators import select_date_range, compute_emas
import plotly.express as px
from plotly.subplots import make_subplots

In [ ]:
try:
    logger
except NameError:
    logger = get_rotating_logger("jupyter", f'logs/daily_screener.log')

def compute_share_turnover(df_volumes, df_shares_outstanding):
    s1 = set(df_shares_outstanding.index)
    s2 = set(df_volumes.columns)
    print('Fidelity - Yahoo:', (s1 - s2), 'Yahoo - Fidelity', (s2 - s1))
    s_so = df_shares_outstanding.loc[list(s1.intersection(s2))].T.iloc[0]
    return df_volumes.div(s_so, axis='columns')

def calc_standing(df):
    df_standing = df.sort_values(by=df.columns[0], ascending=False)
    df_standing = df_standing.reset_index().drop(columns=[c for c  in df.columns if c != 'index'])
    df_standing = df_standing.rename(columns={'index': 'symbol'}).reset_index().set_index('symbol')
    return df_standing + 1

def rank_shareturnover(df_shareturnover):
    indexes = [-1, -5, -20]
    dates = dict([(i, df_shareturnover.index[i].strftime('%F')) for i in indexes])
    df_old_standing = calc_standing(df_shareturnover.iloc[-2:].T).rename(columns={'index': 'old_standing'})
    df_new_standing = calc_standing(df_shareturnover.iloc[-1:].T).rename(columns={'index': 'new_standing'})
    df = df_new_standing.join(df_old_standing)
    df['change'] = df.old_standing - df.new_standing
    return df

def prorate_volumes(df_quotes):
    '''Returns prorated volumes as a series'''
    today = pd.Timestamp.today().normalize()
    _df = df_quotes.set_index('symbol')
    quote_time = pd.to_datetime(today.strftime('%F ') + _df['lastTime'], format='%Y-%m-%d %I:%M:%S%p')
    fraction_day = (quote_time - (today + timedelta(hours=9.5)))/timedelta(hours=6.5)
    fraction_day = fraction_day.apply(lambda x: min(x, 1))
    _df = pd.DataFrame({'volume': _df['volume']/fraction_day}).T
    _df['Date'] = today
    return _df.set_index('Date')

In [ ]:
chain_dir = None
quotes_dir = 'quotes'
cookie_file = 'cookie.txt'
ocd = FidelityOptionDataDownloader(chain_dir, quotes_dir, cookie_file, logger)
ana = OptionAnalyzer('quotes', 'chain')

#### This needs only one update per day.  Make sure the latest Bollinger data are from yesterday
#### Top symbols with highest 20-day volume MA
Notes:
- Volume Avg values are in dollars not shares,
- Volume Std have been normalized to ratios with Volume Avg

In [ ]:
latest_bollinger_file = max(glob('output/bollinger*.csv'))
df_boll = pd.read_csv(latest_bollinger_file)
print('Latest df_boll from file:', latest_bollinger_file, df_boll.shape)
_dfb = df_boll.sort_values(by='Volume_Avg', ascending=False).set_index('symbol').head(df_boll.shape[0]//2)
symlist = list(_dfb.index)
print(f'{len(symlist)} symbols with 20-day average dollar volumes above median:', symlist[:5])
df_close = pd.read_csv('output/yf_close.csv', parse_dates=['Date']).set_index('Date').loc[:, symlist].tail(100)
df_volumes = pd.read_csv('output/yf_volumes.csv', parse_dates=['Date']).set_index('Date').loc[:, symlist].tail(100)
print('Yahoo finance data dates: df_close:', df_close.index[-1].strftime('%F'), 'df_volumes:', df_volumes.index[-1].strftime('%F'))
shares_outstanding_csv_file = os.path.expanduser('~/lab/output/shares_outstanding.csv')
df_shares_outstanding = pd.read_csv(shares_outstanding_csv_file).set_index('symbol')
print('df_shares_outstanding:', df_shares_outstanding.shape)

### Pull latest quotes from fidelity

In [ ]:
_t0 = time.time()
count = 0
batch_size = 40
for idx in range(0, len(symlist), batch_size):
    count += ocd.parallel_get_data(symlist[idx:idx+batch_size], rps=20)
    print('.', end='')
print(count, 'file downloads requested in', int(time.time() - _t0), 'seconds')

In [ ]:
df_quotes, df_shortint, df_vola = ana.get_quote_df(symlist)
df_quotes['symbol'] = df_quotes['symbol'].str.replace('/', '-', regex=False)
print(f'quote time: {df_quotes.lastTime.min()}, {df_quotes.lastTime.max()}, after {int(time.time() - _t0)} seconds')
print('Hopefully no symbol name contains slash:', list(df_quotes[df_quotes.symbol.str.contains('/')].symbol))
dfb = _dfb.join(df_quotes.loc[:, ['symbol', 'lastPrice']].set_index('symbol')).reset_index()
dfb['rank20'] = (dfb['lastPrice'].astype(float) - dfb['MA20'].astype(float))/(dfb['UB20'] - dfb['LB20'])*200
dfb['rank30'] = (dfb['lastPrice'].astype(float) - dfb['MA30'].astype(float))/(dfb['UB30'] - dfb['LB30'])*200
dfb['rank_d'] = dfb.rank20 - dfb.rank30
df_ed = ana.count_days_from_earning_reports(df_quotes)
fig = make_subplots(rows=2)
fig.update_layout(height=400, width=1600)
for _i, chart in enumerate([px.bar(df_ed.head(30), y='earningDays'), px.bar(df_ed.iloc[30:60], y='earningDays')]):
    for trace in chart.data:
        fig.add_trace(trace, row=_i+1, col=1)
fig.show()

### Save shares outstanding data to csv

In [ ]:
df_shares_outstanding = df_quotes.loc[:, ['symbol', 'sharesOutstanding']].set_index('symbol')
df_shares_outstanding.to_csv(shares_outstanding_csv_file)
new_volumes = prorate_volumes(df_quotes)
if new_volumes.index[0] == df_volumes.index[-1]:
    print('New volumes replace the last row of old volumes')
    df_v = pd.concat([df_volumes.iloc[:-1], new_volumes])
else:
    df_v = pd.concat([df_volumes, new_volumes])
df_shareturnover = compute_share_turnover(df_v.tail(20), df_shares_outstanding)
n_tops = 30
top_turnovers = df_shareturnover.tail(1).T.sort_values(by=df_shareturnover.index[-1], ascending=False).head(n_tops)
_title = f'Top {n_tops} Share Turnover Rates - {df_shareturnover.index[-1].strftime('%F')}'
px.bar(df_shareturnover[top_turnovers.index].tail(10), barmode='group', height=600, title=_title).show()
with open('symbols.txt') as fo:
    _symlist = [_.rstrip() for _ in fo]
print(_symlist[:15])
print(_symlist[15:])
print('symbols not in top 25 turnovers:', set(_symlist[15:]) - set(top_turnovers.index))
_df = df_shareturnover.loc[:, [_ for _ in _symlist if _ in df_shareturnover.columns]].tail(10)
_symlist = _df.tail(1).T.sort_values(by=_df.index[-1], ascending=False).index
px.bar(_df.loc[:, _symlist], height=600, barmode='group', title=f'Selected Share Turnover Rates {df_shareturnover.index[-1].strftime('%F')}').show()
rank_shareturnover(df_shareturnover).head(25)

### Bollinger ranks of the top share turnovers

In [ ]:
_dfb_pct = pd.DataFrame([(_, dfb.rank20.quantile(_), dfb.rank30.quantile(_)) for _ in [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]], columns=['pctile', 'rank20', 'rank30'])
px.bar(_dfb_pct, x='pctile', y=['rank20', 'rank30'], barmode='group', width=1000).show()
__df = dfb.set_index('symbol').join(top_turnovers, how='inner').reset_index().rename(columns={'index': 'symbol'}).sort_values(by='rank20')
ana.plot_metrics_in_one_row(__df, ['symbol'], ['rank20', 'rank_d', 'rank30'], shared_y=False, log_y_threshold=-500)

### Selected Bollinger Ranking sorted by 20-day data

In [ ]:
__df = dfb[dfb.symbol.str.contains(r'^(?:SPY|QQQ|GLD|IBIT|DIA|TLT|NVDA|META|MSFT|AAPL|TSLA|AMZN|GOOGL|TSM|AMD|ARM|AVGO|MRVL|HOOD|PLTR|COIN|ETHA|CRCL|ASML|XL|SNDK|LITE|COHR|OKLO)')].sort_values(by='rank20')
ana.plot_metrics_in_one_row(__df, ['symbol'], ['rank20', 'rank_d', 'rank30'], shared_y=False, log_y_threshold=-500)

In [ ]:
__df = dfb.sort_values(by='rank20', ascending=False)
ana.plot_metrics_in_one_row(__df.head(25), ['symbol'], ['rank20', 'rank30'], shared_y=True, log_y_threshold=200)
ana.plot_metrics_in_one_row(__df.tail(25), ['symbol'], ['rank20', 'rank30'], shared_y=True, log_y_threshold=-200)

In [ ]:
df_last_price = df_quotes.loc[:, ['symbol', 'lastPrice']].set_index('symbol').rename(columns={'lastPrice': pd.Timestamp.now().normalize()})
if df_close.index[-1] == df_last_price.T.index[0]:
    df_price = pd.concat([df_close.iloc[:-1], df_last_price.T])
else:
    df_price = pd.concat([df_close, df_last_price.T])

df_ema = compute_emas(df_price)
declining = [_symbol for _symbol in symlist if (lambda x: np.all(x.ema_21 < x.ema_50))(df_ema[_symbol].tail(5))]
ascending = [_symbol for _symbol in symlist if (lambda x: np.all(x.ema_21 > x.ema_50))(df_ema[_symbol].tail(5))]
mixed = [_symbol for _symbol in symlist if (lambda x: ~np.all(x.ema_21 < x.ema_50) & ~np.all(x.ema_21 > x.ema_50))(df_ema[_symbol].tail(5))]
print(len(declining), 'declining,', len(ascending), 'ascending,', len(symlist) - len(declining) - len(ascending), 'limbo')
df_price.iloc[-2:].iloc[:, :15]

In [ ]:
_sym = 'HIMS'#random.sample(ascending, 1)[0]
px.line(df_price.tail(20).loc[:, [_sym]].join(df_ema[_sym].tail(20)), height=600)

In [ ]:
_dfb.join(df_price.iloc[[-1]].T)